# ✈️ Airfoil Noise Prediction using Spark ML

## 📌 Overview

In this project, we build a **machine learning pipeline** using **Apache Spark** to predict airfoil noise levels based on aerodynamic features.

This project simulates a real-world scenario where data engineers and machine learning engineers work together to prepare data, build scalable models, and generate predictions.

To better understand the problem domain, below are simplified diagrams illustrating airflow around an airfoil and the concept of angle of attack.

💡 This project demonstrates an end-to-end Machine Learning pipeline — from raw data processing to model training and prediction using Apache Spark.

---

## 🎯 Objectives

- Perform **ETL operations** on raw airfoil dataset  
- Clean and preprocess the data (remove duplicates and null values)  
- Build a **feature transformation pipeline** using Spark ML  
- Train a **Linear Regression model** for noise prediction  
- Evaluate model performance using multiple metrics  
- Persist and reload the trained model  
- Perform inference on new unseen data  

---

## 🛠️ Technologies Used

- **Apache Spark (PySpark)**
- **Spark ML (Machine Learning Library)**
- **Python**

---

## 📊 Dataset

The dataset contains **1523 observations** related to airfoil performance and noise levels.

Loaded dataset contains **1522 rows**, which may indicate minor inconsistencies in the raw data.

Main features include:

- `Frequency` — frequency of the sound wave  
- `AngleOfAttack` — angle of airflow relative to the wing  
- `ChordLength` — length of the airfoil  
- `FreeStreamVelocity` — speed of the airflow  
- `SuctionSideDisplacement` — displacement on the suction side  

Target variable:

- `SoundLevelDecibels` — noise level (to be predicted)

💡 All input features are numerical, making the dataset suitable for regression modeling using Spark ML.

### Airflow around an airfoil

Diagram of an airfoil. - For informational purpose

![Airfoil Flow](images/airfoil_flow.png)

### Angle of Attack

Diagram showing the Angle of attack. - For informational purpose

![Angle of Attack](images/angle_of_attack.png)

💡 These aerodynamic properties directly influence the generated noise, which is the target variable of our regression model.

---

## 💡 Project Structure

The pipeline consists of the following stages:

1. Data Loading  
2. Data Cleaning and Transformation  
3. Feature Engineering  
4. Machine Learning Pipeline  
5. Model Evaluation  
6. Model Persistence  
7. Inference on New Data  

---

💡 This project demonstrates how **data engineering and machine learning pipelines integrate together** in a scalable Spark environment.

## 1. Import + Spark Initialization

In this section, we initialize the Spark environment and import the required libraries for building a Spark ML regression pipeline.

We use:
- **SparkSession** to work with Spark DataFrames
- **Pipeline** to combine feature engineering and model training steps
- **VectorAssembler** and **StandardScaler** for feature preparation
- **LinearRegression** for prediction
- **RegressionEvaluator** for model evaluation

In [1]:
# Initialize Spark environment

import findspark
findspark.init()

from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.pipeline import PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Create Spark Session

spark = SparkSession \
    .builder \
    .appName("Airfoil Noise Prediction Pipeline") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark session initialized successfully")
print(f"Spark version: {spark.version}")

Spark session initialized successfully
Spark version: 3.5.1


## 2. Data Loading

In this section, we load the raw NASA airfoil noise dataset into a Spark DataFrame.

The dataset is stored in CSV format and contains aerodynamic features used to predict airfoil noise levels.

In [2]:
# Load raw dataset

df = spark.read.csv(
    "data/NASA_airfoil_noise_raw.csv",
    header=True,
    inferSchema=True
)

In [3]:
# Count total number of rows

row_count_raw = df.count()

print("Total rows in raw dataset:", row_count_raw)

Total rows in raw dataset: 1522


The dataset was successfully loaded and contains 1522 rows, which is suitable for building a regression model.

## 3. Schema Inspection

Before transforming the dataset, we inspect its structure and data types.

This helps us:

- Understand the data types of each column  
- Identify potential inconsistencies  
- Prepare the dataset for further transformation and modeling  

In [4]:
# Inspect schema
df.printSchema()

root
 |-- Frequency: integer (nullable = true)
 |-- AngleOfAttack: double (nullable = true)
 |-- ChordLength: double (nullable = true)
 |-- FreeStreamVelocity: double (nullable = true)
 |-- SuctionSideDisplacement: double (nullable = true)
 |-- SoundLevel: double (nullable = true)



In [5]:
# Preview top 5 rows to understand data distribution
df.show(5)

+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows



### 🔍 Initial Observations

- All columns are numerical and suitable for regression modeling  
- The target column is currently named **SoundLevel**  
- No categorical features are present  
- The dataset structure is consistent and ready for cleaning  

💡 Next step: clean the dataset by removing duplicates and null values, and standardize column names.

## 4. Data Cleaning and Transformation

In this section, we clean and prepare the dataset for machine learning.

The main steps include:

- Removing duplicate rows  
- Handling missing (null) values  
- Standardizing column names for clarity  

These steps ensure data quality and improve model performance.

In [6]:
# Count rows before cleaning
rowcount_before = df.count()
print("Rows before cleaning:", rowcount_before)


# Remove duplicate rows
df = df.dropDuplicates()


# Count after removing duplicates
rowcount_after_dedup = df.count()
print("Rows after removing duplicates:", rowcount_after_dedup)


# Remove rows with null values
df = df.dropna()


# Count after removing nulls
rowcount_after_clean = df.count()
print("Rows after removing nulls:", rowcount_after_clean)


# Rename target column for clarity
df = df.withColumnRenamed("SoundLevel", "SoundLevelDecibels")


# Preview cleaned data
df.show(5)


# Check schema after transformation
df.printSchema()

Rows before cleaning: 1522
Rows after removing duplicates: 1503
Rows after removing nulls: 1499
+---------+-------------+-----------+------------------+-----------------------+------------------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevelDecibels|
+---------+-------------+-----------+------------------+-----------------------+------------------+
|     4000|          3.0|     0.3048|              31.7|             0.00529514|           115.608|
|     3150|          2.0|     0.2286|              31.7|             0.00372371|           121.527|
|     2000|          7.3|     0.2286|              31.7|              0.0132672|           115.309|
|     2000|          5.4|     0.1524|              71.3|             0.00401199|           131.111|
|      500|          9.9|     0.1524|              71.3|              0.0193001|           131.279|
+---------+-------------+-----------+------------------+-----------------------+------------------+
only

### 🔍 Cleaning Results

- Duplicate rows have been removed  
- Rows with missing values have been dropped  
- Target column renamed to **SoundLevelDecibels** for clarity  
- Dataset is now clean and ready for feature transformation and modeling  

💡 Clean data ensures better model accuracy and reliability.

The dataset was reduced from 1522 to 1499 rows after cleaning

💾 The cleaned dataset is saved in **Parquet format** for efficient storage and faster processing in downstream tasks.

In [42]:
# Save cleaned dataset in Parquet format (optimized for analytics)
# df.write.mode("overwrite").parquet("output/NASA_airfoil_noise_cleaned")

⚠️ Note: Parquet write operations may fail in local Windows environments due to missing Hadoop configuration.  
This step is provided as a reference for production environments.

💾 For local execution, the dataset is exported as a CSV file.

In [7]:
# Save cleaned dataset as CSV for local development
df.toPandas().to_csv("output/NASA_airfoil_noise_cleaned.csv", index=False)

## 5. Feature Engineering and ML Pipeline

In this section, we build a Spark ML pipeline for regression modeling.

The pipeline includes three main stages:

1. **VectorAssembler** — combines input columns into a single `features` vector  
2. **StandardScaler** — scales numerical features and stores them in `scaledFeatures`  
3. **LinearRegression** — trains a regression model to predict airfoil noise level  

This approach keeps feature transformation and model training inside one reusable machine learning pipeline.

In [9]:
# Define input feature columns

feature_columns = [
    "Frequency",
    "AngleOfAttack",
    "ChordLength",
    "FreeStreamVelocity",
    "SuctionSideDisplacement"
]

# Define VectorAssembler stage

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

# Define StandardScaler stage

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures"
)

# Define Linear Regression model

lr = LinearRegression(
    featuresCol="scaledFeatures",
    labelCol="SoundLevelDecibels"
)

# Build ML pipeline

pipeline = Pipeline(stages=[
    assembler,
    scaler,
    lr
])

### 🔍 Pipeline Setup Summary

- Input features are combined into a single feature vector
- Features are scaled to improve model performance
- A Linear Regression model is configured to predict noise levels

👉 The pipeline is now ready for training and evaluation.

## 6. Train/Test Split and Model Training

In this section, we split the cleaned dataset into training and testing sets.

The model is trained on the training data and later evaluated on unseen testing data.

We use a **70/30 split** with a fixed seed to make the results reproducible.

In [10]:
# Split data into training and testing sets

training_data, testing_data = df.randomSplit([0.7, 0.3], seed=42)

print("Training rows:", training_data.count())
print("Testing rows:", testing_data.count())

Training rows: 1101
Testing rows: 398


In [12]:
# Fit the ML pipeline using training data

pipeline_model = pipeline.fit(training_data)

print("Pipeline model trained successfully")

Pipeline model trained successfully


In [13]:
# Display pipeline stages

for index, stage in enumerate(pipeline.getStages(), start=1):
    print(f"Stage {index}: {stage.__class__.__name__}")

Stage 1: VectorAssembler
Stage 2: StandardScaler
Stage 3: LinearRegression


### 🔍 Training Summary

- The dataset was split into training (1101 rows) and testing (398 rows) sets
- The ML pipeline was successfully trained on the training data
- The pipeline includes feature transformation and a regression model

👉 The model is now ready for evaluation on unseen data.

## 7. Model Evaluation

In this section, we evaluate the performance of the trained regression model.

We use the following metrics:

- **Mean Squared Error (MSE)** — measures average squared difference between predicted and actual values  
- **Mean Absolute Error (MAE)** — measures average absolute difference  
- **R-squared (R²)** — indicates how well the model explains the variance in the data  

These metrics help assess the accuracy and reliability of the model.

In [14]:
# Generate predictions on testing data
predictions = pipeline_model.transform(testing_data)

# Preview predictions
predictions.select("SoundLevelDecibels", "prediction").show(5)

+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376778|
|            133.42|127.37968204568838|
|           119.146|130.34077425074506|
|           116.074|131.11016975113537|
|           134.319|127.12627360125096|
+------------------+------------------+
only showing top 5 rows



In [15]:
# Evaluate MSE
evaluator_mse = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mse"
)

mse = evaluator_mse.evaluate(predictions)


# Evaluate MAE
evaluator_mae = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mae"
)

mae = evaluator_mae.evaluate(predictions)


# Evaluate R2
evaluator_r2 = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)


print(f"MSE: {round(mse, 2)}")
print(f"MAE: {round(mae, 2)}")
print(f"R2: {round(r2, 2)}")

MSE: 25.0
MAE: 3.91
R2: 0.5


### 🔍 Model Evaluation Results

- The model produces reasonable predictions, but some differences between actual and predicted values remain
- **MSE = 25.0** and **MAE = 3.91** indicate the average prediction error
- **R² = 0.50** means the model explains about 50% of the variance in the target variable
- This suggests that the baseline Linear Regression model captures part of the relationship between features and airfoil noise levels

💡 These results provide a solid baseline model, while further improvements could be achieved with additional feature engineering or more advanced regression algorithms.

In [16]:
# Inspect model coefficients
lr_model = pipeline_model.stages[-1]

print("Intercept:", round(lr_model.intercept, 2))

for feature, coef in zip(feature_columns, lr_model.coefficients):
    print(f"{feature}: {round(coef, 4)}")

Intercept: 132.88
Frequency: -3.9906
AngleOfAttack: -2.2881
ChordLength: -3.3269
FreeStreamVelocity: 1.4832
SuctionSideDisplacement: -2.0551


The coefficients show how each feature contributes to the predicted noise level in the linear model.

Positive coefficients increase the predicted value, while negative coefficients decrease it.

## 8. Model Persistence & Export

In this step, we export prediction results and demonstrate how the trained model can be saved for reuse.

For local development, predictions are saved as a CSV file.
Model persistence is included as a production-style reference.

In [17]:
# Save predictions as CSV (local development)
predictions.toPandas().to_csv("output/regression_predictions.csv", index=False)
print("Predictions saved as CSV")


Predictions saved as CSV


In [18]:
# Save trained pipeline model (production-style)
# pipeline_model.write().overwrite().save("output/regression_model")

# Load saved model
# loaded_model = PipelineModel.load("output/regression_model")

📁 The prediction results are exported as a CSV file for local development and analysis.

⚠️ Note:
Model persistence in Spark may require proper Hadoop configuration.
This step is included as a production-style reference.

## 9. Inference Reference

- Demonstrates how the trained pipeline can be reused for predictions
- In production, the saved model can be loaded and used without retraining

💡 Model persistence enables real-world deployment and reuse of trained models.

In [19]:
# Inference example on new unseen data
# This block can be executed in a properly configured Spark environment.

# new_data = spark.createDataFrame([
#     (1000.0, 5.0, 0.02, 30.0, 0.003)
# ], [
#     "Frequency",
#     "AngleOfAttack",
#     "ChordLength",
#     "FreeStreamVelocity",
#     "SuctionSideDisplacement"
# ])

# prediction = pipeline_model.transform(new_data)
# prediction.select("prediction").show()

## 10. Stop Spark Session

In the final step, we gracefully stop the Spark session to release resources.

This is a best practice when working with Spark applications.

In [21]:
# Stop Spark session

spark.stop()

print("Spark session stopped")

Spark session stopped


---
## 11. Conclusion and Insights

## 📊 Model Performance Summary

- The model achieved **MSE = 25.0** and **MAE = 3.91**, indicating moderate prediction error
- The **R² score of 0.50** shows that the model explains about half of the variance in noise levels
- Predictions are reasonably close to actual values, but some deviation remains

## 💡 Key Insights

- The Linear Regression model provides a solid baseline for airfoil noise prediction
- Some features (e.g., Frequency, ChordLength) have a stronger influence on the target variable
- Additional feature engineering or more advanced models could further improve performance

## 🚀 Final Conclusion

This project demonstrates how raw data can be transformed into a fully functional Machine Learning pipeline using Apache Spark.

The pipeline covers:
- Data cleaning and preprocessing
- Feature engineering
- Model training and evaluation
- Model persistence and inference

👉 Overall, the solution reflects a real-world ML workflow and provides a strong foundation for scalable predictive systems.